In [ ]:
import pandas as pd
import re
import glob

# Memuat dan menggabungkan data mentah
all_files = glob.glob('../data/raw/dataset_human_pre2018_strict_indo_7_fix.csv')
print(f"Memuat file dataset: {all_files}")

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)
initial_len = len(df)
print(f"Total data mentah awal: {initial_len} abstrak")

# Menghapus duplikasi data berdasarkan isi teks abstrak
df = df.drop_duplicates(subset=['abstract'], keep='first').copy()
print(f"Jumlah data setelah deduplikasi awal: {len(df)}")


# Pembersihan teks (text cleaning)
# Tanda baca tidak dihapus karena krusial untuk analisis fitur linguistik
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Menghapus entitas HTML
    text = re.sub(r'&[a-zA-Z0-9#]+;', ' ', text)
    
    # Memperbaiki anomali encoding karakter
    text = text.replace('Â', ' ').replace('â€™', "'").replace('â€œ', '"').replace('â€', '"')
    
    # Menyeragamkan karakter whitespace (enter/tab menjadi spasi)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # Menghapus kata pengantar "Abstrak:" atau "Abstract:" di awal kalimat
    text = re.sub(r'^(abstrak|abstract)\s*[-:.]*\s*', '', text, flags=re.IGNORECASE).strip()
    
    # Menghapus spasi ganda
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['abstract'] = df['abstract'].apply(clean_text)

# Melakukan deduplikasi sekunder pasca-pembersihan teks
df = df.drop_duplicates(subset=['abstract'], keep='first').copy()
print(f"Jumlah data setelah pembersihan dan deduplikasi kedua: {len(df)}")

# Penyaringan panjang dokumen
# Mengeliminasi abstrak yang jumlah katanya di bawah 50 kata
df['word_count'] = df['abstract'].apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 50].copy()
df = df.drop(columns=['word_count'])
print(f"Sisa data final (>= 50 kata): {len(df)}")

# Mengekspor dataset bersih tahap 1
df.to_csv('../data/interim/dataset_human_pre2018_cleaned_tahap1_7_fix.csv', index=False, encoding='utf-8')

print(f"\nPra-pemrosesan Tahap 1 selesai. Data diekspor ke dataset_human_pre2018_cleaned_tahap1_7_fix.csv\n")

# Hitung distribusi berdasarkan kolom 'source' (Nama Jurnal)
distribusi = df['source'].value_counts().reset_index()

# Ubah nama kolom biar enak dibaca
distribusi.columns = ['Nama Jurnal', 'Jumlah Abstrak']

# Tambahkan nomor urut mulai dari 1
distribusi.index = distribusi.index + 1

# Tampilkan hasil distribusi
print("Distribusi Hasil Pre-processing Tahap 1 Berdasarkan Jurnal:")
print("-" * 70)
print(distribusi.to_string())
print("-" * 70)

# Tampilkan total keseluruhan
print(f"Total Portal Jurnal yang Berhasil Terambil : {len(distribusi)} Jurnal")
print(f"Total Keseluruhan Abstrak (Final)          : {distribusi['Jumlah Abstrak'].sum()} Abstrak")

Memuat file dataset: ['dataset_human_pre2018_strict_indo_7_fix.csv']
Total data mentah awal: 1139 abstrak
Jumlah data setelah deduplikasi awal: 1139
Jumlah data setelah pembersihan dan deduplikasi kedua: 1139
Sisa data final (>= 50 kata): 1138

Pra-pemrosesan Tahap 1 selesai. Data diekspor ke 'dataset_human_pre2018_cleaned_tahap1_8_fix.csv'

Distribusi Hasil Pre-processing Tahap 1 Berdasarkan Jurnal:
----------------------------------------------------------------------
                                                      Nama Jurnal  Jumlah Abstrak
1   SIMETRIS (Jurnal Teknik Mesin, Elektro dan Ilmu Komputer) UMK             312
2                                        Jurnal INFOTEL IT Telkom             170
3                               JIP (Jurnal Informatika Polinema)             131
4                              Sistemasi (Sistem Informasi) UNISI              82
5                                  Jurnal Sisfokom ISB Atma Luhur              78
6            EXPLORE (Jurnal Sist